# Step 4: Attention

<!-- notebook-only -->

> **How to use this notebook**
> - **Just reading?** The text cells contain the whole lesson. You can skip every code cell. An even nicer way to read: the website, built with `.venv/bin/python website/build.py --serve`.
> - **Want to practice?** Run the code cells from top to bottom (Shift+Enter). They recompute every table and let you experiment. In **Your turn** cells, replace `None` with your answer: `check(...)` says if it is right and gives a hint if not, and `solution("4.1")` shows the worked answer.
> - Every question also has a **Show the answer** box you can open.

<!-- /notebook-only -->

**Goal:** let each token look back at the earlier tokens and take what it needs from them, so that `_want` can find `_hungry`.

In [ ]:
# Run this first (practice only): imports, and our sentence as 8 input vectors (Steps 1 to 3)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from toddler.bpe import BPETokenizer
from toddler.data import CORPUS
from toddler.embeddings import COLUMNS, toddler_table
from toddler.exercises import check, solution
from toddler.positions import POSITION_COLUMNS, with_positions


def short(x):
    return "0" if abs(x) < 1e-9 else f"{x:.4f}".rstrip("0").rstrip(".")


np.set_printoptions(formatter={"float_kind": short})
pd.set_option("display.float_format", short)

STREAM = COLUMNS + POSITION_COLUMNS          # the 10 numbers of every vector
tok = BPETokenizer.train(CORPUS)
table = toddler_table(tok)
ids = tok.encode("I am hungry . I want milk .")
inputs = with_positions(table[ids])          # 8 rows of 10 numbers, as in Step 3
labels = [f"#{p} {tok.vocab[i]}" for p, i in enumerate(ids, start=1)]
pd.DataFrame(inputs, index=labels, columns=STREAM)

## 4.1 The problem: `_want` doesn't know the child is hungry

After Step 3, each vector knows **what** its token is and **where** it is. Here is row 6, `_want`:

```
        me   be want stop feel food  hug sleep    px    py
_want    0    0    1    0    0    0    0    0 -0.71 -0.71
```

Nothing in it says "hungry". But to guess `_milk`, the toddler must know the feeling, 3 tokens back. **Attention** is how a token looks back at the earlier tokens and takes what it needs from them.

## 4.2 The idea: a question, a label and a content

Think of a classroom. `_want` stands up and asks: **"Who is a feeling?"** Every token before it holds up a label, and `_hungry`'s label says **"I'm a feeling!"**. So `_want` listens mostly to `_hungry`, which gives it its content: **"food"**.

Every token makes three small vectors from its own 10 numbers:

| the vector | its job | in our story |
|---|---|---|
| **query** | what am I looking for? | "Who is a feeling?" |
| **key** | what do I have? (my label) | "I'm a feeling!" |
| **value** | what do I give to whoever listens to me? | "food" |

Then each token makes four moves:
1. **Score:** compare its query with the key of every token it can see (dot products, Step 2).
2. **Weights:** turn the scores into shares that add up to 100%.
3. **Mix:** add up the values, each one multiplied by its share.
4. **Add** the mix to its own vector.

## 4.3 Query, key and value: three small tables

Each one is made by multiplying the token's 10 numbers by a table of 10 rows and 4 columns, so queries, keys and values have 4 numbers each. To multiply a vector by a table: multiply each row of the table by the matching number of the vector, then add the rows up.

In a real model, training sets these tables (Step 7). Here we use **our hint**: tables set by hand, almost all zeros.

| table | its only non-zero rows | what it means |
|---|---|---|
| query | row `want` = (6, 0, 0, 0) | only `_want` asks a question |
| key | row `feel` = (2, 0, 0, 0) | only feelings answer it |
| value | row `food` = (1, 0, 0, 0), row `hug` = (0, 1, 0, 0), row `sleep` = (0, 0, 1, 0) | each token offers its topic |

**Worked example: the query of `_want`.** Its vector has want = 1, px = −0.71, py = −0.71, and 0 everywhere else. So:

query = 1 × (row `want`) + (−0.71) × (row `px`) + (−0.71) × (row `py`)
= 1 × (6, 0, 0, 0) + (−0.71) × (0, 0, 0, 0) + (−0.71) × (0, 0, 0, 0) = **(6, 0, 0, 0)**

The same for every token of our sentence:

| pos | token | query | key | value |
|---|---|---|---|---|
| 1 | `_I` | (0, 0, 0, 0) | (0, 0, 0, 0) | (0, 0, 0, 0) |
| 2 | `_am` | (0, 0, 0, 0) | (0, 0, 0, 0) | (0, 0, 0, 0) |
| 3 | `_hungry` | (0, 0, 0, 0) | **(2, 0, 0, 0)** | **(1, 0, 0, 0)**: food |
| 4 | `_.` | (0, 0, 0, 0) | (0, 0, 0, 0) | (0, 0, 0, 0) |
| 5 | `_I` | (0, 0, 0, 0) | (0, 0, 0, 0) | (0, 0, 0, 0) |
| 6 | `_want` | **(6, 0, 0, 0)** | (0, 0, 0, 0) | (0, 0, 0, 0) |
| 7 | `_milk` | (0, 0, 0, 0) | (0, 0, 0, 0) | (1, 0, 0, 0): food |
| 8 | `_.` | (0, 0, 0, 0) | (0, 0, 0, 0) | (0, 0, 0, 0) |

In [ ]:
# Try it: the three hint tables, then the query, key and value of every token
HEAD_SIZE = 4
Wq, Wk, Wv = (np.zeros((len(STREAM), HEAD_SIZE)) for _ in range(3))
Wq[STREAM.index("want"), 0] = 6
Wk[STREAM.index("feel"), 0] = 2
for slot, topic in enumerate(["food", "hug", "sleep"]):
    Wv[STREAM.index(topic), slot] = 1

queries, keys, values = inputs @ Wq, inputs @ Wk, inputs @ Wv
pd.DataFrame({"query": list(queries), "key": list(keys), "value": list(values)}, index=labels)

## 4.4 Scores: the question meets every label

**score = (query · key) ÷ √4**

The ÷ √4 (that is, ÷ 2) keeps the scores from growing too big when the vectors get long. Real models also divide by the square root of their key size.

A token can only look at itself and at the tokens **before** it. When the toddler guesses what comes after `_want`, `_milk` hasn't been said yet, so it is hidden. This is called the **causal mask**: no peeking at the future.

The scores of `_want` (#6):

| token it looks at | #1 `_I` | #2 `_am` | #3 `_hungry` | #4 `_.` | #5 `_I` | #6 `_want` |
|---|---|---|---|---|---|---|
| query · key | 6 × 0 = 0 | 0 | 6 × 2 = 12 | 0 | 0 | 0 |
| ÷ 2 = score | 0 | 0 | **6** | 0 | 0 | 0 |

## 4.5 From scores to shares: softmax

The scores must become shares that add up to 100%. The recipe is called **softmax**, and we will meet it again in Step 6:

1. Raise e (2.718…) to the power of each score. The result is always positive, and a bigger score gives a much bigger result.
2. Divide each result by the total.

| | #1 | #2 | #3 `_hungry` | #4 | #5 | #6 | total |
|---|---|---|---|---|---|---|---|
| score | 0 | 0 | 6 | 0 | 0 | 0 | |
| e^score | 1 | 1 | 403.4 | 1 | 1 | 1 | 408.4 |
| ÷ 408.4 = weight | 0.24% | 0.24% | **98.8%** | 0.24% | 0.24% | 0.24% | 100% |

`_want` gives **98.8%** of its attention to `_hungry`.

![_want looks back: the weights, with the key and the value of each token](figures/04_want_looks_back.png)

In [ ]:
# Try it: the scores of _want, with the future hidden, then softmax
def softmax(scores):
    exps = np.exp(scores - scores.max())  # subtracting the max avoids huge numbers and changes nothing
    return exps / exps.sum()


want = 5                                  # row 5 is position 6: Python counts from 0
visible = slice(0, want + 1)              # positions 1 to 6: itself and the tokens before it
want_scores = queries[want] @ keys[visible].T / np.sqrt(HEAD_SIZE)
print("e^score:", np.exp(want_scores), "  total:", short(np.exp(want_scores).sum()))
pd.DataFrame({"score": want_scores, "weight": softmax(want_scores)}, index=labels[visible])

## 4.6 Mixing the values, and adding the result

Each value is multiplied by its weight, and the results are added up:

mix = 0.0024 × (0, 0, 0, 0) [`_I`] + 0.0024 × (0, 0, 0, 0) [`_am`] + 0.988 × (1, 0, 0, 0) [`_hungry`] + … = **(0.988, 0, 0, 0)**

A fourth small table, the head's **output table** (4 rows, 10 columns), puts the mix back into the 10 columns. In our hint, slot 1 goes to the `food` column, slot 2 to `hug` and slot 3 to `sleep`. So the mix becomes +0.988 in the `food` column, and it is **added** to `_want`'s vector:

```
              me   be want stop feel  food  hug sleep    px    py
_want          0    0    1    0    0     0    0    0 -0.71 -0.71
+ attention    0    0    0    0    0  0.99    0    0     0     0
= new _want    0    0    1    0    0  0.99    0    0 -0.71 -0.71
```

`_want` now carries "food": the child wants something about food. In sentence 2, `_want` finds `_sad` and takes `hug`; in sentence 3, it finds `_tired` and takes `sleep`.

In [ ]:
# Try it: mix the values, put the mix back into the 10 columns, and add it to _want
Wo = np.zeros((HEAD_SIZE, len(STREAM)))
for slot, topic in enumerate(["food", "hug", "sleep"]):
    Wo[slot, STREAM.index(topic)] = 1

mix = softmax(want_scores) @ values[visible]
added = mix @ Wo
print("mix:", mix)
pd.DataFrame([inputs[want], added, inputs[want] + added], index=["_want", "+ attention", "= new _want"],
             columns=STREAM)

## 4.7 Every token at once

All 8 tokens make the same moves at the same time. With tables, the whole sentence is computed in one go: all the queries, all the keys, a table of 8 × 8 scores, the mask on the future, softmax on each row, then the mix.

![Who looks at whom in our sentence](figures/04_attention_pattern.png)

- Row 6: `_want` looks at `_hungry` (99%).
- Row 1: the first `_I` can only see itself, so it gives itself 100%.
- The other tokens ask no question in our hint: their query is 0, so all their scores are 0, and they spread their attention evenly over what they can see. For example, `_hungry` (#3) takes ⅓ of each of #1, #2 and #3, so it adds ⅓ × food to its own vector. That's harmless here.

In [ ]:
# Try it: the whole sentence at once
def attention(inputs, Wq, Wk, Wv, Wo):
    queries, keys, values = inputs @ Wq, inputs @ Wk, inputs @ Wv
    scores = queries @ keys.T / np.sqrt(Wq.shape[1])
    scores[np.triu(np.ones(scores.shape, dtype=bool), k=1)] = -np.inf   # the future is hidden: e^-inf = 0
    exps = np.exp(scores - scores.max(axis=1, keepdims=True))
    weights = exps / exps.sum(axis=1, keepdims=True)
    return weights @ values @ Wo, weights


added_all, weights = attention(inputs, Wq, Wk, Wv, Wo)
pd.DataFrame(weights, index=labels, columns=[f"#{p}" for p in range(1, 9)])

In [ ]:
# Try it: in each of the 3 sentences, what attention adds to _want
for sentence in CORPUS:
    sentence_added, _ = attention(with_positions(table[tok.encode(sentence)]), Wq, Wk, Wv, Wo)
    gained = {STREAM[i]: short(v) for i, v in enumerate(sentence_added[5]) if abs(v) > 1e-9}
    print(f"{sentence:<30} → _want gets {gained}")

## 4.8 Several heads, looking for different things

One set of query, key, value and output tables is called a **head**. Real models run many heads side by side, each one looking for something different, and add up what they bring.

For example, a head can look at **the previous token**, using the clock of Step 3:
- its query is the token's own clock hand, **turned back by 45°** (one step);
- its key is the clock hand itself.

The score is highest for the token whose hand points exactly where the turned-back hand points: the previous token. For `_want` (#6, at 225°), the hand turned back points at 180°, which is #5. With hands made 16 times longer (a sharper question), this head gives 84% of its attention to the previous token, everywhere in the sentence. The cell below computes it.

Our toddler uses a single head, the "feeling finder". Real models have dozens of heads in each block.

In [ ]:
# Try it: a second head that looks one step back, with the clock
turn = np.radians(45)
Wq_back, Wk_back = np.zeros((len(STREAM), HEAD_SIZE)), np.zeros((len(STREAM), HEAD_SIZE))
Wq_back[STREAM.index("px"), :2] = 16 * np.array([np.cos(turn), -np.sin(turn)])   # turns the hand back by 45°
Wq_back[STREAM.index("py"), :2] = 16 * np.array([np.sin(turn), np.cos(turn)])
Wk_back[STREAM.index("px"), 0] = 1
Wk_back[STREAM.index("py"), 1] = 1
_, weights_back = attention(inputs, Wq_back, Wk_back, Wv, Wo)
pd.DataFrame(weights_back, index=labels, columns=[f"#{p}" for p in range(1, 9)])

## 4.9 How real LLMs do it

- **Sizes:** queries and keys of 64 or 128 numbers, and dozens of heads in each block (GPT-3: 96 heads in each of its 96 blocks).
- **The mask** is the same: when it writes, a model never looks at the tokens that come after.
- **The cost:** every token looks at every earlier token, so the work grows like length × length. That's one reason why context windows have a limit. While writing, models keep the keys and values of the earlier tokens in memory instead of computing them again (the **KV cache**).
- **Positions:** most recent models turn queries and keys by the clock angle right inside attention (RoPE, Step 3), so the scores depend on the distance between tokens.
- What each head looks for is **learned**, never written by hand. Researchers find heads that look at the previous token, at the matching bracket, at the name mentioned earlier…

## 4.10 What is still missing

`_want` now carries "food", but nothing has decided "→ milk" yet. Step 5 builds the rest of the transformer block (the feed-forward part, and why everything is *added*). Step 6 turns the final vector into a probability for each token.

<!-- notebook-only -->

The same code is saved in `toddler/attention.py`, and the next notebooks import it from there.

<!-- /notebook-only -->

In [ ]:
# Try it: the package version used by the next notebooks gives the same result
from toddler import attention as toddler_attention

package_added, package_weights = toddler_attention.attention(inputs, **toddler_attention.hint_attention())
assert np.allclose(package_added, added_all) and np.allclose(package_weights, weights)
print("toddler/attention.py gives the same attention")

## Your turn

Do each one **by hand first**, using the tables above.

### 4.1 If the query row of `want` were (2, 0, 0, 0) instead of (6, 0, 0, 0), what share of its attention would `_want` give to `_hungry`?

<details>
<summary>Show the answer</summary>
<!-- answer: 0.6 | 0.60 | 0.596 | 0.5964 | 60% | 59.6% -->
<!-- format: a number between 0 and 1, e.g. 0.25 -->

The score of `_hungry` becomes 2 × 2 ÷ 2 = **2**, and the 5 other scores stay 0.

e² = 7.39, so the total is 7.39 + 5 × 1 = 12.39, and the share is 7.39 ÷ 12.39 = **0.60**.

A weaker question gives a less focused attention: 60% instead of 98.8%.

</details>

In [ ]:
my_answer = None  # a number between 0 and 1
check("4.1", my_answer)

### 4.2 In "I am sad . I want mommy .", which column of `_want`'s vector does attention add to?

<details>
<summary>Show the answer</summary>
<!-- answer: hug | _hug -->
<!-- format: a column name, e.g. sleep -->

`_want` finds the feeling `_sad` (its key is (2, 0, 0, 0)). The value of `_sad` is (0, 1, 0, 0): slot 2, which the output table puts in the **hug** column. So `_want` gets +0.99 hug.

</details>

In [ ]:
my_answer = None  # a column name, e.g. "sleep"
check("4.2", my_answer)

### 4.3 `_hungry` (#3) has the query (0, 0, 0, 0). What share of its attention does it give to each token it can see?

<details>
<summary>Show the answer</summary>
<!-- answer: 1/3 | 0.33 | 0.333 | 33% | 33.3% -->
<!-- format: a fraction or a number, e.g. 1/4 -->

All its scores are 0, so all the e^score are 1. It can see 3 tokens (#1, #2 and #3), so each one gets 1 ÷ 3 = **⅓**.

</details>

In [ ]:
my_answer = None  # a number, e.g. 0.25
check("4.3", my_answer)

<!-- notebook-only -->

### 4.4 (code) Write `my_softmax(scores)`: e to the power of each score, divided by the total.

<details>
<summary>Show the answer</summary>

```python
def my_softmax(scores):
    exps = np.exp(scores)
    return exps / exps.sum()
```

</details>

<!-- /notebook-only -->

In [ ]:
def my_softmax(scores):
    # your code here
    return None


check("4.4", my_softmax)

### 4.5 Think: why can't `_hungry` (#3) look at `_milk` (#7)?

<details>
<summary>Show the answer</summary>
<!-- choices: *`_milk` comes later: when the toddler writes, the future doesn't exist yet, so it is hidden | `_milk`'s key is 0 | `_hungry` has no query | `_milk` is too far away -->

Because of the **causal mask**. When the toddler writes, it guesses one token at a time: the tokens after the current one don't exist yet. Training must work the same way, so every token only sees itself and the tokens before it.

</details>

### 4.6 Think: why are the scores divided by √4?

<details>
<summary>Show the answer</summary>
<!-- choices: *To keep the scores from growing too big with long vectors, so that softmax doesn't get too sharp | To make the scores positive | To make the computation faster | Because a query has 2 parts -->

A dot product adds up one product per number, so it tends to grow with the size of the vectors. Big scores make softmax almost all-or-nothing, and learning gets hard. Dividing by √(key size) keeps the scores in a good range, whatever the size.

</details>

## Summary

- **Attention** lets each token look back at the earlier tokens and take what it needs.
- Each token makes a **query** (what am I looking for?), a **key** (what do I have?) and a **value** (what do I give?), with three small tables.
- score = query · key ÷ √(key size). The future is hidden (the **causal mask**). **Softmax** turns the scores into shares that add up to 100%. The values are mixed with these shares, and the result is **added** to the token's vector.
- With our hint tables, `_want` gives 98.8% of its attention to `_hungry`, and gets +0.99 food.
- One set of these tables is a **head**. Real models run many heads at once.
- Next: **Step 5, the transformer block**.